# 04b — Logistic Regression: Patient Analysis

Models MRSA acquisition as a function of prior antibiotic-class exposure, in the
**patient** cohort. This cohort is matched on ward, admission month/year, and room LOS —
**not** age or sex — so unlike `04a`, age and sex are added as explicit covariates here to
avoid confounding.

To make that necessity concrete, this notebook fits the model both **without** and
**with** the age/sex adjustment and compares the antibiotic-exposure coefficients.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

pat = load_processed("patient_mrsa.csv")
pat.shape

## Build design matrix

In [ ]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]

pat_enc = pat.copy()
pat_enc["sex_male"] = (pat_enc["sex"].str.lower() == "male").astype(int)

base_predictors = abx_cols + ["elix_index_mortality"]
adjusted_predictors = base_predictors + ["age", "sex_male"]

model_df = pat_enc[["group_binary"] + adjusted_predictors].dropna()
print(model_df.shape)
model_df.describe()

## Multicollinearity check (VIF, adjusted model)

In [ ]:
X = sm.add_constant(model_df[adjusted_predictors])
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif

## Unadjusted model (antibiotics only, no age/sex)

In [ ]:
X_unadj = sm.add_constant(model_df[base_predictors])
y = model_df["group_binary"]

logit_pat_unadj = sm.Logit(y, X_unadj).fit()
logit_pat_unadj.summary()

## Adjusted model (antibiotics + age + sex)

In [ ]:
X_adj = sm.add_constant(model_df[adjusted_predictors])

logit_pat_adj = sm.Logit(y, X_adj).fit()
logit_pat_adj.summary()

## Compare antibiotic-exposure coefficients before/after adjustment

In [ ]:
comparison = pd.DataFrame({
    "unadjusted_OR": np.exp(logit_pat_unadj.params[base_predictors]),
    "adjusted_OR": np.exp(logit_pat_adj.params[base_predictors]),
})
comparison["pct_shift"] = (
    (comparison["adjusted_OR"] - comparison["unadjusted_OR"]) / comparison["unadjusted_OR"] * 100
)
comparison.sort_values("pct_shift", key=abs, ascending=False)

## Interpretation

A meaningful shift in the antibiotic-class odds ratios between the unadjusted and
age/sex-adjusted models confirms age/sex confounding was live in this cohort (as expected,
since the patient analysis wasn't matched on them) — the **adjusted** model is the one to
report and carry into `05_model_comparison.ipynb`.

In [ ]:
import pickle
Path("../reports").mkdir(exist_ok=True)
with open("../reports/logit_patient_adjusted.pkl", "wb") as f:
    pickle.dump(logit_pat_adj, f)